# Olist Brazilian E-Commerce — Starter Notebook

About 100,000 real, anonymized orders (2016–2018) from Olist, a Brazilian online marketplace that connects small sellers to customers. Each order has a purchase date, delivery dates, the seller, the customer's state, prices, and usually a **1–5 star review**.

The data come as several linked tables. This notebook joins them into **one row per order** so you can start exploring.

**How this notebook is organized**

| Section | What it does |
|---|---|
| Settings | Where the data lives (the only cell you might edit) |
| Load Data | Imports libraries, loads and prepares the data |
| Inspect the Dataset | First rows, size, column types |
| Describe Variables | Summary statistics |
| Check Missing Data | What's missing and why it matters |
| Basic Visualizations | A few plots to get oriented |
| Variables You May Want to Explore | Outcomes, predictors, grouping and time variables |
| Your Analysis Starts Here | Your space for HW 1, HW 2, and HW 3 |

Cells marked 🧰 are **starter code** written by the instructor. Cells marked ✏️ are **yours**.

Open the table of contents (the ☰ icon on the left in Colab) to jump between sections.

## Settings

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

If the default link stops working, or you downloaded the file yourself, change the path below. See the dataset guide for where to get the data.

In [ ]:
# Settings: where the data lives.
# By default the data load from this course's GitHub repository, so there's nothing to download.
REPO = "cogs118d/datasets"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/01_olist/"

# Cloned the repo and working on your own computer? Use the local folder instead:
# BASE_URL = "../data/01_olist/"
# Saved the files to Google Drive? Point BASE_URL at that folder, e.g.:
# BASE_URL = "/content/drive/MyDrive/01_olist/"

# Load Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Libraries used in every COGS 118D starter notebook
import numpy as np                        # numbers and arrays
import pandas as pd                       # data tables ("DataFrames")
import matplotlib.pyplot as plt           # plots
import statsmodels.formula.api as smf     # GLMs and mixed models (you'll use this later)

pd.set_option("display.max_columns", 60)  # show more columns when printing tables
plt.rcParams["figure.figsize"] = (7, 4)   # default plot size

In [ ]:
# Load the five tables we need (this takes about 30 seconds)
orders    = pd.read_csv(BASE_URL + "olist_orders_dataset.csv")
items     = pd.read_csv(BASE_URL + "olist_order_items_dataset.csv")
sellers   = pd.read_csv(BASE_URL + "olist_sellers_dataset.csv")
customers = pd.read_csv(BASE_URL + "olist_customers_dataset.csv")
reviews   = pd.read_csv(BASE_URL + "olist_order_reviews_dataset.csv")

for name, table in [("orders", orders), ("items", items), ("sellers", sellers), ("customers", customers), ("reviews", reviews)]:
    print(f"{name:10s} {table.shape}")

## Preprocessing: join the tables into one row per order

An order can contain several items, possibly from different sellers, and a few orders have more than one review. We summarize items per order and keep the first review, so each order appears once.

In [ ]:
# 1. Items: one row per order (number of items, total price, total shipping, first seller)
items_per_order = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    price=("price", "sum"),
    freight=("freight_value", "sum"),
    seller_id=("seller_id", "first"),
    n_sellers=("seller_id", "nunique"),
).reset_index()

# 2. Reviews: keep the first review for each order
reviews_first = (reviews.sort_values("review_creation_date")
                 .drop_duplicates("order_id")[["order_id", "review_score"]])

# 3. Join everything onto the orders table
df = (orders
      .merge(items_per_order, on="order_id", how="left")
      .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
      .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
      .merge(reviews_first, on="order_id", how="left"))
print("One row per order:", df.shape)

In [ ]:
# 4. Dates: convert text to real dates, then compute a few useful quantities
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[col] = pd.to_datetime(df[col])

df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
df["promised_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
df["late"] = np.where(df["order_delivered_customer_date"].notna(),
                      (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(float), np.nan)
df["cross_state"] = (df["seller_state"] != df["customer_state"]).astype(int)  # shipped between states?
df["low_review"] = np.where(df["review_score"].notna(), (df["review_score"] <= 2).astype(float), np.nan)
df[["order_status", "delivery_days", "late", "cross_state", "review_score", "low_review"]].head()

# Inspect the Dataset

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# The first 5 rows: each row is one observation
df.head()

In [ ]:
# Dimensions: (number of rows, number of columns)
print("Rows, columns:", df.shape)

In [ ]:
# Column names and data types (numbers, text, dates, True/False)
df.info()

In [ ]:
# Order status: most orders were delivered
df["order_status"].value_counts()

# Describe Variables

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Numeric summaries
df[["n_items", "price", "freight", "delivery_days", "promised_days", "review_score"]].describe().round(2)

In [ ]:
# Review scores (1-5 stars)
df["review_score"].value_counts().sort_index()

# Check Missing Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Count and percent of missing values in each column (only columns with any missing)
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "percent": (100 * missing / len(df)).round(1)})
missing_table[missing_table["n_missing"] > 0].sort_values("n_missing", ascending=False)

**What to notice.** Delivery dates are missing for orders that were never delivered (canceled, unavailable, or still shipped when the data were collected). About 1% of orders have no review. A few orders have no items. Decide how to handle each of these, and say so in your write-up.

# Basic Visualizations

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

These plots are for getting oriented. They are not answers to any homework question.

In [ ]:
# Distribution of review scores
counts = df["review_score"].value_counts().sort_index()
plt.bar(counts.index, counts.values)
plt.xlabel("Review score (stars)")
plt.ylabel("Number of orders")
plt.title("Review scores")
plt.show()

In [ ]:
# How long delivery takes (delivered orders; x-axis cut at 60 days to see the bulk)
plt.hist(df["delivery_days"].dropna().clip(upper=60), bins=60, edgecolor="white")
plt.xlabel("Days from purchase to delivery (values above 60 shown at 60)")
plt.ylabel("Number of orders")
plt.title("Delivery time")
plt.show()

In [ ]:
# Delivery time for same-state vs. cross-state orders
delivered = df[df["delivery_days"].notna()]
groups = [delivered.loc[delivered["cross_state"] == 0, "delivery_days"],
          delivered.loc[delivered["cross_state"] == 1, "delivery_days"]]
plt.boxplot(groups, showfliers=False)
plt.xticks([1, 2], ["Same state", "Different states"])
plt.ylabel("Days to delivery")
plt.title("Delivery time by shipping route (outliers hidden)")
plt.show()

In [ ]:
# Orders by customer state (top 10)
df["customer_state"].value_counts().head(10).plot(kind="bar")
plt.ylabel("Number of orders")
plt.title("Where customers are")
plt.show()

# Variables You May Want to Explore

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

**Outcome variables:** `review_score` (1–5), `low_review` (1–2 stars, yes/no), `delivery_days`, `late` (yes/no).

**Predictors:** `late`, `delivery_days`, `promised_days`, `cross_state`, `price`, `freight`, `n_items`, `customer_state`, `seller_state`, purchase date.

In [ ]:
outcome_vars = ["review_score", "low_review", "delivery_days", "late"]
predictor_vars = ["cross_state", "price", "freight", "n_items", "promised_days"]
df[outcome_vars + predictor_vars].describe().round(2)

## Grouping structure (for mixed-effects models in HW 3)

Orders are grouped within **sellers** and within **customer states**. Customers almost never order twice, so customers are *not* a useful grouping variable.

In [ ]:
orders_per_seller = df.groupby("seller_id").size()
print("Sellers:", orders_per_seller.shape[0])
print(orders_per_seller.describe().round(1))
print()
print("Customer states:", df["customer_state"].nunique())
print("Customers with more than one order:", (df.groupby("customer_unique_id").size() > 1).sum())

## Time-to-event variables (for survival analysis in HW 3)

**Appropriate.** *Time* = days from purchase until the order reaches the customer. *Event* = delivered. Orders still marked `shipped` at the end of data collection never reached the customer while we watched, so they are **censored** at the last date in the data. Canceled or unavailable orders are left out, because they end in a different way.

In [ ]:
surv = df[df["order_status"].isin(["delivered", "shipped"])].copy()
end_of_data = df["order_purchase_timestamp"].max()
surv["event_delivered"] = surv["order_delivered_customer_date"].notna().astype(int)
surv["time_days"] = np.where(surv["event_delivered"] == 1, surv["delivery_days"],
                             (end_of_data - surv["order_purchase_timestamp"]).dt.total_seconds() / 86400)

print("Orders in the survival table:", len(surv))
print("Delivered (event = 1):", surv["event_delivered"].sum(), "  Censored (event = 0):", (surv["event_delivered"] == 0).sum())
surv[["time_days", "event_delivered", "cross_state"]].head()

---
# Your Analysis Starts Here

> ✏️ **Your analysis.** Everything below this line is yours. Add as many cells as you need.

The dataset guide lists starter questions for each homework. The `surv` table is ready for HW 3 if you choose survival analysis.

## HW 1 — Describe

In [ ]:
# ✏️ Your HW 1 code here


## HW 2 — Predict / Moderate / Mediate

In [ ]:
# ✏️ Your HW 2 code here


## HW 3 — Mixed Effects / Survival

In [ ]:
# ✏️ Your HW 3 code here
